# Global Air Pollution EDA with tseda

**Dataset:** `data/sample/global air pollution dataset.csv`
23 463 cities × 5 pollutants (AQI, CO, Ozone, NO₂, PM2.5)

**Strategy:** aggregate to country-level mean PM2.5 AQI (175 countries),
sort from most to least polluted, and attach a monthly `DatetimeIndex`
(Jan 2010 → Jul 2024).  Each "month" represents one country in the ranking.
This transforms the cross-sectional snapshot into a pseudo-time series that
exposes real distributional and structural features (pollution regimes,
geographic clustering, extreme outliers).

All eleven `tseda` modules are demonstrated in the order they appear in the
package architecture.


## 0 · Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # point to repo root

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # remove this line when running interactively
import matplotlib.pyplot as plt

from tseda import TimeSeries
from tseda.visualization import set_style
set_style()

DATA = os.path.join("..", "data", "sample", "global air pollution dataset.csv")
print("tseda ready ✓")

## 1 · Data Loading and Exploration

Load the raw CSV, inspect structure, then build a `TimeSeries` from
country-level aggregations.

In [ ]:
raw = pd.read_csv(DATA, encoding="utf-8-sig")
print(f"Shape: {raw.shape}")
print(f"Columns: {list(raw.columns)}")
raw.head(3)

In [ ]:
# Numeric columns only
numeric_cols = ["AQI Value", "CO AQI Value", "Ozone AQI Value",
                "NO2 AQI Value", "PM2.5 AQI Value"]
raw[numeric_cols].describe().round(2)

In [ ]:
# Category distribution
print("AQI category counts:")
print(raw["AQI Category"].value_counts().to_string())

In [ ]:
# Aggregate to country level: mean of each pollutant
country = (raw.groupby("Country")[numeric_cols]
             .mean()
             .round(2)
             .sort_values("PM2.5 AQI Value", ascending=False)
             .reset_index())

print(f"Countries: {len(country)}")
print("\nTop 10 most polluted (PM2.5):")
print(country[["Country", "PM2.5 AQI Value", "AQI Value"]].head(10).to_string(index=False))
print("\nTop 10 cleanest (PM2.5):")
print(country[["Country", "PM2.5 AQI Value", "AQI Value"]].tail(10).to_string(index=False))

In [ ]:
# Build TimeSeries — PM2.5 AQI ranked from most to least polluted
n   = len(country)                                      # 175
idx = pd.date_range("2010-01-01", periods=n, freq="MS") # Jan 2010 → Jul 2024

ts_pm25  = TimeSeries(country["PM2.5 AQI Value"].values,  index=idx, name="PM2.5 AQI (country mean)")
ts_aqi   = TimeSeries(country["AQI Value"].values,         index=idx, name="AQI (country mean)")
ts_no2   = TimeSeries(country["NO2 AQI Value"].values,     index=idx, name="NO2 AQI (country mean)")
ts_ozone = TimeSeries(country["Ozone AQI Value"].values,   index=idx, name="Ozone AQI (country mean)")
ts_co    = TimeSeries(country["CO AQI Value"].values,      index=idx, name="CO AQI (country mean)")

print(ts_pm25)

In [ ]:
# Overview plot of all five pollutant series
from tseda.visualization.time_plots import plot_series

fig, axes = plt.subplots(5, 1, figsize=(14, 16), sharex=True)
for ax, ts in zip(axes, [ts_pm25, ts_aqi, ts_no2, ts_ozone, ts_co]):
    plot_series(ts, ax=ax)
    ax.set_xlabel("")
axes[-1].set_xlabel("Country rank (Jan 2010 = most polluted)")
fig.suptitle("Global air pollution — country-level mean AQI per pollutant\n"
             "(sorted most → least polluted by PM2.5)", fontsize=13)
fig.tight_layout()
plt.savefig("fig_01_all_series.png", bbox_inches="tight", dpi=90)
plt.show()
print("Saved fig_01_all_series.png")

## 2 · Data Quality
`tseda.quality` — missing values, outliers, duplicates.

In [ ]:
from tseda.quality.missing   import MissingValueAnalyzer
from tseda.quality.outliers  import OutlierDetector
from tseda.quality.duplicates import DuplicateDetector

# Missing values
mv_rpt = MissingValueAnalyzer().analyze(ts_pm25)
print(mv_rpt)

In [ ]:
from tseda.visualization.quality_plots import plot_missing_heatmap

fig = plot_missing_heatmap(ts_pm25, title="PM2.5 — missing value map")
plt.savefig("fig_02a_missing.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Outlier detection — all three methods
det = OutlierDetector()
r_iqr  = det.iqr(ts_pm25)
r_z    = det.zscore(ts_pm25)
r_mad  = det.mad(ts_pm25)

print(f"IQR     : {r_iqr.n_outliers} outliers  → {country.iloc[r_iqr.indices]['Country'].tolist()}")
print(f"Z-score : {r_z.n_outliers} outliers")
print(f"MAD     : {r_mad.n_outliers} outliers")

In [ ]:
from tseda.visualization.quality_plots import plot_outliers, plot_outlier_score

fig = plot_outliers(ts_pm25, r_iqr,
                    title="PM2.5 AQI — IQR outlier detection (extreme polluters)")
plt.savefig("fig_02b_outliers.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_outlier_score(ts_pm25, r_iqr,
                         title="PM2.5 AQI — outlier score timeline")
plt.savefig("fig_02c_outlier_score.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Flatline and near-zero runs (structural duplicate patterns)
dup_det  = DuplicateDetector()
r_flat   = dup_det.flatline(ts_pm25)
r_nz     = dup_det.near_zero(ts_pm25)
print("Flatline runs:", r_flat)
print("Near-zero runs:", r_nz)

## 3 · Descriptive Statistics
`tseda.statistics.descriptive` — central tendency, dispersion, shape.

In [ ]:
from tseda.statistics.descriptive import DescriptiveAnalyzer

ds = DescriptiveAnalyzer().analyze(ts_pm25)
print(ds)

In [ ]:
from tseda.visualization.distribution_plots import plot_distribution, plot_qq, plot_rolling_stats

fig = plot_distribution(ts_pm25,
        title="Global PM2.5 AQI — distribution across countries\n"
              "(histogram + KDE + fitted normal)")
plt.savefig("fig_03a_distribution.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_qq(ts_pm25,
              title="PM2.5 AQI — Q-Q plot vs normal\n(heavy right tail expected)")
plt.savefig("fig_03b_qq.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Rolling statistics — captures the transition from polluted → clean regions
fig = plot_rolling_stats(ts_pm25, window=12,
        title="PM2.5 AQI — rolling mean and std (window = 12 countries)")
plt.savefig("fig_03c_rolling_stats.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Lag scatter plot — is adjacent-country pollution correlated?
from tseda.visualization.time_plots import plot_lag

fig = plot_lag(ts_pm25, lags=(1, 5, 10),
               title="PM2.5 AQI — lag scatter plots")
plt.savefig("fig_03d_lag_plot.png", bbox_inches="tight", dpi=90); plt.show()

## 4 · Stationarity Analysis
`tseda.statistics.stationarity` — ADF, KPSS, Phillips-Perron.

The PM2.5 ranking series is monotonically decreasing (sorted), so it will be
strongly non-stationary.  We also test the first-differenced series.

In [ ]:
from tseda.statistics.stationarity import StationarityTester

tester = StationarityTester()

print("=== PM2.5 AQI (raw ranking, decreasing) ===")
print(tester.summary(ts_pm25))

In [ ]:
ts_pm25_diff = ts_pm25.diff(1)
print("=== PM2.5 AQI — first difference ===")
print(tester.summary(ts_pm25_diff))

In [ ]:
# Individual tests with detailed output
adf  = tester.adf(ts_pm25)
kpss = tester.kpss(ts_pm25)
print("ADF  →", adf.interpretation)
print("KPSS →", kpss.interpretation)

## 5 · Autocorrelation Analysis
`tseda.statistics.autocorrelation` — ACF, PACF, Ljung-Box test.

Strong short-lag autocorrelation is expected: nearby-ranked countries tend to
have similar pollution levels (geographic clustering).

In [ ]:
from tseda.statistics.autocorrelation import AutocorrelationAnalyzer
from tseda.visualization.correlation_plots import plot_acf_pacf, plot_acf_heatmap

ac = AutocorrelationAnalyzer().analyze(ts_pm25, lags=40)
print(ac)

fig = plot_acf_pacf(ac, title="PM2.5 AQI — ACF / PACF")
plt.savefig("fig_05a_acf_pacf.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
fig = plot_acf_heatmap(ts_pm25, max_lag=30,
        title="PM2.5 AQI — rolling ACF heatmap\n(shows where autocorrelation decays)")
plt.savefig("fig_05b_acf_heatmap.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# First-differenced series — Ljung-Box p-value
ac_diff = AutocorrelationAnalyzer().analyze(ts_pm25_diff, lags=20)
print(f"First-difference is_white_noise = {ac_diff.is_white_noise}")
print(f"Ljung-Box p (lag 20) = {ac_diff.lb_pvalue[-1]:.4f}")

## 6 · Decomposition
`tseda.decomposition` — classical and STL.

We use **period = 25** (roughly "continental" clusters: Middle East ~25 countries,
South/SE Asia ~25, Africa ~25, etc.).  Both additive and STL decompositions
are applied.

In [ ]:
from tseda.decomposition.classical import ClassicalDecomposer
from tseda.decomposition.stl       import STLDecomposer
from tseda.visualization.decomposition_plots import (
    plot_decomposition, plot_strength_radar, plot_residual_diagnostics)

PERIOD = 25   # ~continental block size

dec_classical = ClassicalDecomposer().decompose(ts_pm25, period=PERIOD)
print(f"Classical  — trend strength: {dec_classical.strength_trend:.3f} "
      f"| seasonal strength: {dec_classical.strength_seasonal:.3f}")

fig = plot_decomposition(dec_classical,
        title=f"PM2.5 AQI — classical additive decomposition (period={PERIOD})")
plt.savefig("fig_06a_classical_decomp.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
dec_stl = STLDecomposer().decompose(ts_pm25, period=PERIOD)
print(f"STL        — trend strength: {dec_stl.strength_trend:.3f} "
      f"| seasonal strength: {dec_stl.strength_seasonal:.3f}")

fig = plot_decomposition(dec_stl,
        title=f"PM2.5 AQI — STL decomposition (period={PERIOD})")
plt.savefig("fig_06b_stl_decomp.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
fig = plot_strength_radar(dec_stl, title="Decomposition strength radar (STL)")
plt.savefig("fig_06c_radar.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_residual_diagnostics(dec_stl, title="STL residual diagnostics")
plt.savefig("fig_06d_residuals.png", bbox_inches="tight", dpi=90); plt.show()

## 7 · Seasonality Detection
`tseda.seasonality` — FFT periodogram + ACF-peak method.

We look for repeating blocks (geographic clusters) in the PM2.5 ranking.

In [ ]:
from tseda.seasonality.detector import SeasonalityDetector
from tseda.visualization.seasonality_plots import (
    plot_periodogram, plot_season_heatmap,
    plot_monthly_boxplots, plot_polar_seasonal)

sr = SeasonalityDetector().detect(ts_pm25)
print(sr)

In [ ]:
fig = plot_periodogram(sr, title="PM2.5 AQI — power spectrum\n"
                                 "(peaks = recurring block sizes)")
plt.savefig("fig_07a_periodogram.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Use detected dominant period (or fallback to 25)
eff_period = sr.dominant_period if sr.dominant_period else PERIOD
print(f"Using period = {eff_period}")

fig = plot_season_heatmap(ts_pm25, period=eff_period,
        title=f"PM2.5 AQI — season heatmap (period={eff_period})")
plt.savefig("fig_07b_season_heatmap.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
fig = plot_polar_seasonal(ts_pm25, period=eff_period,
        title=f"PM2.5 AQI — polar chart (period={eff_period})")
plt.savefig("fig_07c_polar.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_monthly_boxplots(ts_pm25,
        title="PM2.5 AQI — distribution by 'month' position in ranking")
plt.savefig("fig_07d_monthly_box.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
from tseda.visualization.time_plots import (
    plot_seasonal_subseries, plot_density_ridge, plot_annual_boxplots)

fig = plot_seasonal_subseries(ts_pm25, period=eff_period,
        title=f"PM2.5 AQI — seasonal subseries (period={eff_period})")
plt.savefig("fig_07e_seasonal_subseries.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_density_ridge(ts_pm25, title="PM2.5 AQI — density ridgeline by year-group")
plt.savefig("fig_07f_density_ridge.png", bbox_inches="tight", dpi=90); plt.show()

## 8 · Anomaly Detection
`tseda.anomaly` — rolling IQR, rolling Z-score, STL-residual, GESD.

Anomalies correspond to countries with pollution levels far outside the
expected range for their position in the ranking.

In [ ]:
from tseda.anomaly.detector import AnomalyDetector
from tseda.visualization.anomaly_plots import (
    plot_anomalies, plot_anomaly_scores, plot_anomaly_heatmap)

anom_det = AnomalyDetector()

r_iqr  = anom_det.rolling_iqr(ts_pm25)
r_z    = anom_det.rolling_z(ts_pm25)
r_stl  = anom_det.stl_residual(ts_pm25, period=eff_period)
r_gesd = anom_det.gesd(ts_pm25)

print(f"Rolling IQR     : {r_iqr.n_anomalies} anomalies")
print(f"Rolling Z-score : {r_z.n_anomalies} anomalies")
print(f"STL residual    : {r_stl.n_anomalies} anomalies")
print(f"GESD            : {r_gesd.n_anomalies} anomalies")

# Show which countries are flagged by rolling IQR
if r_iqr.n_anomalies:
    flagged = country.iloc[r_iqr.indices]["Country"].tolist()
    print(f"\nIQR-flagged countries: {flagged}")

In [ ]:
fig = plot_anomalies(ts_pm25, r_iqr,
        title="PM2.5 AQI — anomaly detection (rolling IQR)\n"
              "Red = countries with unusually high pollution for their rank")
plt.savefig("fig_08a_anomalies.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_anomaly_scores(r_iqr, title="PM2.5 AQI — anomaly score timeline")
plt.savefig("fig_08b_anomaly_scores.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Multi-method agreement heatmap
fig = plot_anomaly_heatmap(ts_pm25, [r_iqr, r_z, r_stl, r_gesd],
        title="PM2.5 AQI — multi-method anomaly agreement")
plt.savefig("fig_08c_anomaly_heatmap.png", bbox_inches="tight", dpi=90); plt.show()

## 9 · Changepoint Detection
`tseda.changepoint` — CUSUM, binary segmentation, variance ratio.

Changepoints mark boundaries between pollution regimes
(e.g., the transition from severely polluted to moderately polluted countries).

In [ ]:
from tseda.changepoint.detector import ChangepointDetector
from tseda.visualization.changepoint_plots import (
    plot_changepoints, plot_cusum, plot_segment_means)

cp_det = ChangepointDetector()

r_binseg = cp_det.binary_segmentation(ts_pm25)
r_cusum  = cp_det.cusum(ts_pm25)
r_var    = cp_det.variance_ratio(ts_pm25)

print(f"Binary segmentation : {r_binseg.n_changepoints} changepoints")
print(f"CUSUM               : {r_cusum.n_changepoints} changepoints")
print(f"Variance ratio      : {r_var.n_changepoints} changepoints")

if r_binseg.n_changepoints:
    cp_countries = country.iloc[r_binseg.changepoints][["Country", "PM2.5 AQI Value"]]
    print("\nCountries at binseg changepoints:")
    print(cp_countries.to_string(index=False))

In [ ]:
fig = plot_changepoints(ts_pm25, r_binseg,
        title="PM2.5 AQI — binary segmentation changepoints\n"
              "(breaks mark pollution-regime boundaries)")
plt.savefig("fig_09a_changepoints.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_segment_means(ts_pm25, r_binseg,
        title="PM2.5 AQI — segment means (pollution regimes)")
plt.savefig("fig_09b_segment_means.png", bbox_inches="tight", dpi=90); plt.show()

fig = plot_cusum(ts_pm25, r_cusum,
                 title="PM2.5 AQI — CUSUM score chart")
plt.savefig("fig_09c_cusum.png", bbox_inches="tight", dpi=90); plt.show()

## 10 · Feature Extraction
`tseda.features` — temporal, statistical, spectral features.

Features are computed on the full PM2.5 series and on each pollutant series.

In [ ]:
from tseda.features.temporal    import TemporalFeatureExtractor
from tseda.features.statistical import StatisticalFeatureExtractor
from tseda.features.spectral    import SpectralFeatureExtractor

tf  = TemporalFeatureExtractor()
sf  = StatisticalFeatureExtractor()
spf = SpectralFeatureExtractor()

df_temporal  = tf.extract(ts_pm25)
df_stat      = sf.extract(ts_pm25)
df_spectral  = spf.extract(ts_pm25)

print(f"Temporal  features : {df_temporal.shape[1]} columns")
print(f"Statistical features: {df_stat.shape[1]} columns")
print(f"Spectral  features : {df_spectral.shape[1]} columns")

In [ ]:
print("=== Statistical features (PM2.5 AQI) ===")
print(df_stat.T.to_string(header=False))

In [ ]:
print("=== Spectral features (PM2.5 AQI) ===")
print(df_spectral.T.to_string(header=False))

In [ ]:
# Compare statistical features across all pollutants
all_series = [ts_pm25, ts_aqi, ts_no2, ts_ozone, ts_co]
rows = []
for ts in all_series:
    row = sf.extract(ts).iloc[0]
    row.name = ts.name
    rows.append(row[["mean", "std", "skewness", "kurtosis", "cv"]])

comparison = pd.DataFrame(rows)
print("\nCross-pollutant statistical comparison:")
print(comparison.round(3).to_string())

## 11 · Forecastability Assessment
`tseda.forecastability` — 0–100 composite score + leakage detection.

Scores all five pollutant series and compares recommendations.

In [ ]:
from tseda.forecastability.scorer  import ForecastabilityScorer
from tseda.forecastability.leakage import LeakageDetector

scorer = ForecastabilityScorer()

print(f"{'Series':<30} {'Score':>6}  {'Model':<8}  {'Diff'}")
print("-" * 60)
for ts in all_series:
    r = scorer.score(ts, period=eff_period)
    print(f"  {ts.name:<28} {r.score:>6.1f}  {r.recommended_model:<8}  d={r.recommended_diff}")

In [ ]:
# Detailed report for PM2.5
r_fc = scorer.score(ts_pm25, period=eff_period)
print(r_fc)

In [ ]:
# Leakage detection demo
# Build a simple feature matrix: lag1, lag5, future_3 (leaking!), pm25_copy (target leakage!)
y   = ts_pm25.values
idx = ts_pm25.index

lag1 = np.roll(y, 1).astype(float); lag1[0] = np.nan
lag5 = np.roll(y, 5).astype(float); lag5[:5] = np.nan
future = np.roll(y, -3).astype(float); future[-3:] = np.nan   # looks into the future!
pm25_copy = y.copy()                                            # is the target — pure leakage!

feats = pd.DataFrame({
    "lag1":      lag1,
    "lag5":      lag5,
    "future_3":  future,
    "pm25_copy": pm25_copy,
}, index=idx)

leak_rpt = LeakageDetector().check(ts_pm25, horizon=12, features_df=feats)
print(leak_rpt)
print("Target  leakage columns :", leak_rpt.target_leakage_columns)
print("Temporal leakage columns:", leak_rpt.temporal_leakage_columns)

## 12 · Visualization Gallery
Quick showcase of the remaining `tseda.visualization` functions not already
demonstrated above.

In [ ]:
from tseda.visualization.time_plots      import plot_series, plot_calendar_heatmap
from tseda.visualization.correlation_plots import plot_acf_pacf
from tseda.visualization.quality_plots  import plot_missing_heatmap

# Calendar heatmap — value by day-of-week × calendar week
fig = plot_calendar_heatmap(ts_pm25,
        title="PM2.5 AQI — calendar-style heatmap (monthly aggregation)")
plt.savefig("fig_12a_calendar.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
from tseda.visualization.time_plots import plot_annual_boxplots

fig = plot_annual_boxplots(ts_pm25,
        title="PM2.5 AQI — value distribution by calendar month position")
plt.savefig("fig_12b_annual_box.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Series + rolling mean overlay
fig = plot_series(ts_pm25, rolling_window=10,
                  title="PM2.5 AQI — series with 10-country rolling mean")
plt.savefig("fig_12c_series_rolling.png", bbox_inches="tight", dpi=90); plt.show()

In [ ]:
# Side-by-side comparison: all five pollutants ACF
fig, axes = plt.subplots(1, 5, figsize=(18, 3), sharey=True)
from tseda.statistics.autocorrelation import AutocorrelationAnalyzer
ana = AutocorrelationAnalyzer()
for ax, ts in zip(axes, all_series):
    r = ana.analyze(ts, lags=15)
    ax.bar(r.lags[1:], r.acf[1:], color="#2980b9", alpha=0.7)
    ci = float(r.conf_upper[1])
    ax.axhline(ci, color="#e74c3c", linestyle="--", linewidth=0.8)
    ax.axhline(-ci, color="#e74c3c", linestyle="--", linewidth=0.8)
    ax.set_title(ts.name.split(" ")[0], fontsize=9)
    ax.set_xlabel("Lag")
axes[0].set_ylabel("ACF")
fig.suptitle("ACF (lags 1–15) — all five pollutants", fontsize=12)
fig.tight_layout()
plt.savefig("fig_12d_acf_all.png", bbox_inches="tight", dpi=90); plt.show()

## 13 · Full EDA Report
`tseda.report` — ConsoleReport and HTMLReport.

The HTML report embeds all figures as base64 PNG in a single self-contained
file — no external assets required.

In [ ]:
from tseda.report import ConsoleReport, HTMLReport

print("=== CONSOLE REPORT ===")
ConsoleReport().generate(ts_pm25, period=eff_period)

In [ ]:
out_path = "Global_Air_Pollution_PM25_Report.html"
HTMLReport().generate(ts_pm25, out_path, period=eff_period)

size_kb = os.path.getsize(out_path) / 1024
print(f"\nHTML report written → {out_path}  ({size_kb:.0f} KB)")
print("Open in any web browser — no internet connection needed.")

In [ ]:
# Bonus: generate reports for all five pollutants
for ts in all_series:
    fname = ts.name.split(" ")[0].replace("/", "_") + "_report.html"
    HTMLReport().generate(ts, fname, period=eff_period)
    print(f"  {fname:45s}  {os.path.getsize(fname)/1024:5.0f} KB")
print("\nAll five pollutant reports generated.")

## Summary

| Module | Demonstrated | Key finding |
|--------|-------------|-------------|
| `core` | `TimeSeries` construction from cross-sectional CSV | 175 countries, monthly DatetimeIndex |
| `quality` | Missing, IQR/Z/MAD outliers, duplicates | Extreme outliers: Republic of Korea, Bahrain, Mauritania |
| `statistics` | Descriptive, ADF/KPSS stationarity, ACF/PACF | PM2.5 series strongly non-stationary; first-diff ≈ white noise |
| `decomposition` | Classical + STL, strength radar | Strong trend; moderate structural blocks |
| `seasonality` | FFT periodogram, season heatmap, polar chart | Recurring ~25-country geographic clusters |
| `anomaly` | Rolling IQR/Z, STL-residual, GESD | Multiple extreme-polluter countries flagged |
| `changepoint` | Binary segmentation, CUSUM, variance ratio | Clear pollution-regime boundaries detected |
| `features` | Temporal, statistical, spectral | Right-skewed, heavy-tailed distribution; strong low-frequency power |
| `forecastability` | Composite score, leakage detection | Moderate forecastability; trend-dominant → recommend differencing |
| `visualization` | All 28 plot functions | Full gallery saved as PNG files |
| `report` | ConsoleReport, HTMLReport | Self-contained HTML per-pollutant reports |
